# 🛰️ Step 1: Preprocessing & LightGBM Pipeline (v3)
### Encroachment Detection on Resourcesat LISS-4 (5m Resolution)

**Improvements in v3:**
1. **TOA Reflectance Conversion** using exact `BAND_META.txt` calibration parameters.
2. **Sub-pixel Co-registration** via phase cross-correlation.
3. **Pseudo-Invariant Feature (PIF)** relative radiometric normalization.
4. **Sub-pixel Footprint Alignment** (0.25 px refinement).
5. **31 Engineered Spatial Features** including MBI, PanTex, road distance, and multi-temporal deltas.
6. **Out-of-Fold Label Noise Filtering** to eliminate unmapped buildings in training negatives.
7. **LightGBM Boosted Decision Trees** + Continuous Footprint Coverage Regressor.

In [ ]:
# 1. Environment & Dependencies Setup
import os, sys

# Mount Google Drive if running in Google Colab
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT_DIR = "/content/drive/MyDrive/Encroachment"
    if not os.path.isdir(ROOT_DIR):
        ROOT_DIR = "/content/drive/My Drive/Encroachment"
    sys.path.append(ROOT_DIR)
    %pip install -q rasterio geopandas lightgbm scikit-learn scipy shapely optuna
else:
    ROOT_DIR = os.getcwd()
    sys.path.append(ROOT_DIR)

print("Working Root:", ROOT_DIR)

In [ ]:
# 2. Initialize Central Configuration and Check Files
import numpy as np
from encroach.config import Config
from encroach.scene import Scene

cfg = Config(ROOT=ROOT_DIR)
# Override layer paths here if your shapefiles are stored elsewhere:
# cfg.AOI_SHP = os.path.join(ROOT_DIR, "layers", "aoi_ring_15_200m.shp")
# cfg.BUILD_SHP = os.path.join(ROOT_DIR, "layers", "buildings_ring_15_200m.shp")

cfg.check()
rng = np.random.default_rng(cfg.SEED)

In [ ]:
# 3. Initialize Scene: TOA Radiometry, Sub-pixel Co-registration, PIF Normalization & Footprint Alignment
scene = Scene(cfg, load_state=False)
scene.prepare(rng)
print("Scene preparation complete. Calibrated state saved to scene_state.json.")

In [ ]:
# 4. Pass 1: Footprint Coverage Raster & Spatial Block Split
from encroach.labels import coverage_pass, spatial_split

cnt = coverage_pass(scene)
split = spatial_split(scene, cnt, rng)

In [ ]:
# 5. Pass 2: Feature Extraction & Pixel Sampling
from encroach.sampling import sample_pixels

data = sample_pixels(scene, split, cnt, rng)
Xtr, ytr, bid_tr = data["Xtr"], data["ytr"], data["bid_tr"]
Xva, yva = data["Xva"], data["yva"]
Xte, yte = data["Xte"], data["yte"]
Xmix, cov_mix = data["Xmix"], data["cov_mix"]

In [ ]:
# 6. Clean Label Noise (Out-Of-Fold False Negative Removal)
from encroach.gbm import clean_label_noise

if cfg.CLEAN_LABELS:
    Xtr_clean, ytr_clean, _ = clean_label_noise(Xtr, ytr, bid_tr, clean_thr=cfg.CLEAN_THR, seed=cfg.SEED)
else:
    Xtr_clean, ytr_clean = Xtr, ytr

In [ ]:
# 7. Train LightGBM Classifier & Continuous Coverage Regressor
from encroach.gbm import train_classifier, train_regressor, find_optimal_threshold, save_gbm_models

clf = train_classifier(Xtr_clean, ytr_clean, Xva, yva, seed=cfg.SEED)
reg = train_regressor(Xmix, cov_mix, seed=cfg.SEED)

best_thr, val_metrics = find_optimal_threshold(clf, Xva, yva)
save_gbm_models(cfg, clf, reg, val_metrics)